# SmartSentry AML — Inference Orchestrator

Runs the pipeline against a **new input file** (same schema as the output of `01__aml_typology_detector`, but **without** `is_aml`, `aml_typology`, or `typology_group_id` columns).

Stages executed:
1. **02 Rules Engine** — apply 126 compliance rules
2. **03 Feature Engineering** — compute velocity, balance, graph features
3. **04 Phase 1 (predict)** — load saved Phase 1 model and score
4. **05 Phase 2 (predict)** — load saved Phase 2 model and classify typology

Final output: `predictions_output.parquet` in the Phase 2 output directory.


## 1 — Configuration

In [1]:
import os
import sys
import time
import json
import subprocess
from datetime import datetime, timedelta
from pathlib import Path

# ═══════════════════════════════════════════════════════════════
# CONFIG — update paths to match your environment
# ═══════════════════════════════════════════════════════════════

NOTEBOOK_DIR = os.getcwd()
OUTPUT_DIR   = os.path.join(os.path.dirname(NOTEBOOK_DIR), "outputs_updated")
PHASE1_DIR   = os.path.join(os.path.dirname(NOTEBOOK_DIR), "python_scripts", "ml_outputs")
PHASE2_DIR   = os.path.join(os.path.dirname(NOTEBOOK_DIR), "python_scripts", "phase2_outputs")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── Input file: the user-provided transaction file ───────────
# Override at runtime by setting AML_INFERENCE_INPUT env var.
DEFAULT_INPUT = os.path.join(OUTPUT_DIR, "inference_input.parquet")
INPUT_FILE = os.environ.get("AML_INFERENCE_INPUT", DEFAULT_INPUT)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Inference input file not found: {INPUT_FILE}\n"
        f"Set AML_INFERENCE_INPUT env var or place file at default path."
    )

TIMEOUT_MINUTES = 60
STOP_ON_FAILURE = True
SAVE_EXECUTED_NOTEBOOKS = True

# Verify saved model bundles exist
required_bundles = [
    os.path.join(PHASE1_DIR, "phase1_model_bundle.joblib"),
    os.path.join(PHASE2_DIR, "phase2_model_bundle.joblib"),
]
print("=" * 70)
print("SmartSentry AML — Inference Orchestrator")
print("=" * 70)
print(f"  Input file:        {INPUT_FILE}")
print(f"  Phase 1 bundle:    {required_bundles[0]}")
print(f"  Phase 2 bundle:    {required_bundles[1]}")
print(f"  Working directory: {NOTEBOOK_DIR}")
print(f"  Output directory:  {OUTPUT_DIR}")

for b in required_bundles:
    if not os.path.exists(b):
        raise FileNotFoundError(
            f"Required model bundle missing: {b}\n"
            f"Train the models first using 00__aml_pipeline_orchestrator with yes."
        )
    print(f"  ✓ {os.path.basename(b)}: {os.path.getsize(b)/(1024*1024):.2f} MB")

print()


SmartSentry AML — Inference Orchestrator
  Input file:        c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated\inference_input.parquet
  Phase 1 bundle:    c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\ml_outputs\phase1_model_bundle.joblib
  Phase 2 bundle:    c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\phase2_outputs\phase2_model_bundle.joblib
  Working directory: c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts
  Output directory:  c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated
  ✓ phase1_model_bundle.joblib: 5.07 MB
  ✓ phase2_model_bundle.joblib: 22.03 MB



## 2 — Inference Pipeline Definition

In [2]:
# Notebooks to run, in order. Each runs in its own kernel.
# AML_RUN_MODE=predict, AML_INPUT_FILE chains through the pipeline.

INFERENCE_PIPELINE = [
    {
        "id":          "02",
        "name":        "Rules Engine",
        "notebook":    "02__aml_rules_engine.ipynb",
        "description": "Apply 126 RBI/PMLA compliance rules",
        "input_env":   INPUT_FILE,
        "output_file": os.path.join(OUTPUT_DIR, "stg_transactions_rules.parquet"),
    },
    {
        "id":          "03",
        "name":        "Feature Engineering",
        "notebook":    "03__aml_feature_engineering.ipynb",
        "description": "Compute velocity, balance, graph features",
        "input_env":   os.path.join(OUTPUT_DIR, "stg_transactions_rules.parquet"),
        "output_file": os.path.join(OUTPUT_DIR, "stg_transactions_features.parquet"),
    },
    {
        "id":          "04",
        "name":        "Phase 1 (predict)",
        "notebook":    "04__aml_ml_preparation.ipynb",
        "description": "Score transactions with Phase 1 binary AML model",
        "input_env":   os.path.join(OUTPUT_DIR, "stg_transactions_features.parquet"),
        "output_file": os.path.join(PHASE1_DIR, "df_ml_phase_1.parquet"),
    },
    {
        "id":          "05",
        "name":        "Phase 2 (predict)",
        "notebook":    "05__aml_phase2_typology_classifier.ipynb",
        "description": "Classify into 10 typologies; multi-label output",
        # Phase 2 reads df_ml_phase_1.parquet from PHASE1_DIR via env var.
        # Set input_env to the same so AML_INPUT_FILE is always a valid path string.
        "input_env":   os.path.join(PHASE1_DIR, "df_ml_phase_1.parquet"),
        "output_file": os.path.join(PHASE2_DIR, "predictions_output.parquet"),
    },
]

# Defensive: every stage MUST have a string input_env. None values come from
# stale kernel state and produce env var "None" → garbage path → confusing errors.
for stage in INFERENCE_PIPELINE:
    if not stage["input_env"] or not isinstance(stage["input_env"], str):
        raise ValueError(
            f"Stage {stage['id']} has invalid input_env={stage['input_env']!r}. "
            f"Re-run this cell from scratch."
        )

print("Verifying notebooks present...")
for stage in INFERENCE_PIPELINE:
    nb_path = os.path.join(NOTEBOOK_DIR, stage["notebook"])
    status = "✓" if os.path.exists(nb_path) else "⚠ NOT FOUND"
    print(f"  [{stage['id']}] {stage['notebook']:<55s} {status}")
    if not os.path.exists(nb_path):
        raise FileNotFoundError(stage["notebook"])
print("  All notebooks present.\n")

# Sanity-print the chain
print("Pipeline chain (input → notebook → output):")
for stage in INFERENCE_PIPELINE:
    print(f"  [{stage['id']}] {os.path.basename(stage['input_env']):<45s} → {stage['notebook']:<45s} → {os.path.basename(stage['output_file'])}")

Verifying notebooks present...
  [02] 02__aml_rules_engine.ipynb                              ✓
  [03] 03__aml_feature_engineering.ipynb                       ✓
  [04] 04__aml_ml_preparation.ipynb                            ✓
  [05] 05__aml_phase2_typology_classifier.ipynb                ✓
  All notebooks present.

Pipeline chain (input → notebook → output):
  [02] inference_input.parquet                       → 02__aml_rules_engine.ipynb                    → stg_transactions_rules.parquet
  [03] stg_transactions_rules.parquet                → 03__aml_feature_engineering.ipynb             → stg_transactions_features.parquet
  [04] stg_transactions_features.parquet             → 04__aml_ml_preparation.ipynb                  → df_ml_phase_1.parquet
  [05] df_ml_phase_1.parquet                         → 05__aml_phase2_typology_classifier.ipynb      → predictions_output.parquet


## 3 — Notebook Runner

In [3]:
def run_notebook_with_env(notebook_path, env_overrides, timeout_minutes=60, working_dir=None):
    """Execute a notebook in a fresh kernel, with env variables set."""
    start_time = time.time()
    notebook_name = os.path.basename(notebook_path)

    executed_dir = os.path.join(OUTPUT_DIR, "executed_inference_notebooks")
    os.makedirs(executed_dir, exist_ok=True)
    executed_path = os.path.join(executed_dir, notebook_name)

    env = os.environ.copy()
    env.update({str(k): str(v) for k, v in env_overrides.items()})

    print(f"  Executing: {notebook_name}")
    for k, v in env_overrides.items():
        print(f"    {k}={v}")

    try:
        cmd = [
            sys.executable, "-m", "jupyter", "nbconvert",
            "--to", "notebook", "--execute",
            "--ExecutePreprocessor.timeout=" + str(timeout_minutes * 60),
            "--ExecutePreprocessor.kernel_name=python3",
            "--output", executed_path,
            notebook_path,
        ]
        result = subprocess.run(
            cmd, capture_output=True, text=True,
            timeout=timeout_minutes * 60 + 60,
            cwd=working_dir or os.path.dirname(notebook_path),
            env=env,
        )
        elapsed = time.time() - start_time
        if result.returncode == 0:
            return {"status": "SUCCESS", "elapsed_seconds": elapsed,
                    "elapsed_str": str(timedelta(seconds=int(elapsed))),
                    "stdout": result.stdout[-500:] if result.stdout else ""}
        else:
            return {"status": "FAILED", "elapsed_seconds": elapsed,
                    "elapsed_str": str(timedelta(seconds=int(elapsed))),
                    "error": (result.stderr or "")[-1500:],
                    "stdout": (result.stdout or "")[-500:]}
    except subprocess.TimeoutExpired:
        elapsed = time.time() - start_time
        return {"status": "TIMEOUT", "elapsed_seconds": elapsed,
                "elapsed_str": str(timedelta(seconds=int(elapsed))),
                "error": f"exceeded {timeout_minutes} min"}
    except Exception as e:
        elapsed = time.time() - start_time
        return {"status": "ERROR", "elapsed_seconds": elapsed,
                "elapsed_str": str(timedelta(seconds=int(elapsed))),
                "error": str(e)}

print("Runner loaded.")


Runner loaded.


## 4 — Execute Inference Pipeline

In [4]:
print("=" * 75)
print("INFERENCE PIPELINE EXECUTION STARTED")
print(f"  Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

pipeline_start = time.time()
pipeline_results = []
pipeline_failed = False

for stage in INFERENCE_PIPELINE:
    print(f"\n{'─' * 70}")
    print(f"  STAGE [{stage['id']}] {stage['name']}")
    print(f"  {stage['description']}")
    print(f"{'─' * 70}")

    if pipeline_failed and STOP_ON_FAILURE:
        print("  ⊘ SKIPPED (previous stage failed)")
        pipeline_results.append({"stage": stage["id"], "name": stage["name"],
                                  "status": "SKIPPED", "elapsed_str": "—",
                                  "elapsed_seconds": 0})
        continue

   # Defensive: input_env must be a real, existing string path.
    # If it's None, empty, or "None", the subprocess would receive a literal
    # "None" as AML_INPUT_FILE and pandas would error obscurely.
    stage_input = stage.get("input_env")
    if not stage_input or stage_input in ("None", "none", None) or not isinstance(stage_input, str):
        print(f"  ✗ Stage [{stage['id']}] has invalid input_env={stage_input!r} — aborting")
        pipeline_failed = True
        pipeline_results.append({"stage": stage["id"], "name": stage["name"],
                                  "status": "ERROR", "elapsed_str": "—",
                                  "elapsed_seconds": 0})
        continue
    if not os.path.exists(stage_input):
        print(f"  ✗ Stage [{stage['id']}] input file does not exist: {stage_input}")
        print(f"     Previous stage may have completed without writing the expected output.")
        pipeline_failed = True
        pipeline_results.append({"stage": stage["id"], "name": stage["name"],
                                  "status": "ERROR", "elapsed_str": "—",
                                  "elapsed_seconds": 0})
        continue

    env_overrides = {
        "AML_RUN_MODE":    "predict",
        "AML_INPUT_FILE":  stage_input,
        "AML_PHASE1_DIR":  PHASE1_DIR,
        "AML_PHASE2_DIR":  PHASE2_DIR,
    }
    
    nb_path = os.path.join(NOTEBOOK_DIR, stage["notebook"])
    result = run_notebook_with_env(nb_path, env_overrides,
                                    timeout_minutes=TIMEOUT_MINUTES,
                                    working_dir=NOTEBOOK_DIR)

    status_icon = {"SUCCESS": "✓", "FAILED": "✗", "TIMEOUT": "⏱", "ERROR": "⚠"}.get(result["status"], "?")
    print(f"\n  {status_icon} Status: {result['status']} ({result['elapsed_str']})")

    if result["status"] != "SUCCESS":
        print(f"  Error: {(result.get('error') or '')[:400]}")
        pipeline_failed = True
    else:
        out = stage["output_file"]
        if os.path.exists(out):
            sz = os.path.getsize(out) / (1024 * 1024)
            print(f"  Output:  ✓ {os.path.basename(out):<50s} {sz:>8.2f} MB")
        else:
            print(f"  Output:  ⚠ MISSING: {out}")
            pipeline_failed = True

    pipeline_results.append({"stage": stage["id"], "name": stage["name"],
                              "notebook": stage["notebook"], "status": result["status"],
                              "elapsed_str": result["elapsed_str"],
                              "elapsed_seconds": result["elapsed_seconds"]})

pipeline_elapsed = time.time() - pipeline_start
print(f"\n{'=' * 70}")
print("INFERENCE PIPELINE COMPLETE")
print(f"  Total time: {str(timedelta(seconds=int(pipeline_elapsed)))}")
print(f"  End time:   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'=' * 70}")


INFERENCE PIPELINE EXECUTION STARTED
  Start time: 2026-05-15 20:47:26

──────────────────────────────────────────────────────────────────────
  STAGE [02] Rules Engine
  Apply 126 RBI/PMLA compliance rules
──────────────────────────────────────────────────────────────────────
  Executing: 02__aml_rules_engine.ipynb
    AML_RUN_MODE=predict
    AML_INPUT_FILE=c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated\inference_input.parquet
    AML_PHASE1_DIR=c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\ml_outputs
    AML_PHASE2_DIR=c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\phase2_outputs



  ✓ Status: SUCCESS (0:01:34)
  Output:  ✓ stg_transactions_rules.parquet                         7.73 MB

──────────────────────────────────────────────────────────────────────
  STAGE [03] Feature Engineering
  Compute velocity, balance, graph features
──────────────────────────────────────────────────────────────────────
  Executing: 03__aml_feature_engineering.ipynb
    AML_RUN_MODE=predict
    AML_INPUT_FILE=c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated\stg_transactions_rules.parquet
    AML_PHASE1_DIR=c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\ml_outputs
    AML_PHASE2_DIR=c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\phase2_outputs



  ✓ Status: SUCCESS (0:06:01)
  Output:  ✓ stg_transactions_features.parquet                     17.99 MB

──────────────────────────────────────────────────────────────────────
  STAGE [04] Phase 1 (predict)
  Score transactions with Phase 1 binary AML model
──────────────────────────────────────────────────────────────────────
  Executing: 04__aml_ml_preparation.ipynb
    AML_RUN_MODE=predict
    AML_INPUT_FILE=c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated\stg_transactions_features.parquet
    AML_PHASE1_DIR=c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\ml_outputs
    AML_PHASE2_DIR=c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\phase2_outputs



  ✓ Status: SUCCESS (0:00:16)
  Output:  ✓ df_ml_phase_1.parquet                                 19.02 MB

──────────────────────────────────────────────────────────────────────
  STAGE [05] Phase 2 (predict)
  Classify into 10 typologies; multi-label output
──────────────────────────────────────────────────────────────────────
  Executing: 05__aml_phase2_typology_classifier.ipynb
    AML_RUN_MODE=predict
    AML_INPUT_FILE=c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\ml_outputs\df_ml_phase_1.parquet
    AML_PHASE1_DIR=c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\ml_outputs
    AML_PHASE2_DIR=c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\phase2_outputs



  ✓ Status: SUCCESS (0:00:18)
  Output:  ✓ predictions_output.parquet                             3.29 MB

INFERENCE PIPELINE COMPLETE
  Total time: 0:08:11
  End time:   2026-05-15 20:55:37


## 5 — Summary

In [5]:
print("\n" + "=" * 70)
print("STAGE SUMMARY")
print("=" * 70)
print(f"  {'Stage':<6s} {'Name':<25s} {'Status':<10s} {'Duration':<12s}")
print(f"  {'─' * 60}")
total_ok = 0
for r in pipeline_results:
    icon = {"SUCCESS": "✓", "FAILED": "✗", "TIMEOUT": "⏱", "ERROR": "⚠", "SKIPPED": "⊘"}.get(r["status"], "?")
    print(f"  [{r['stage']}]  {r['name']:<25s} {icon} {r['status']:<8s} {r['elapsed_str']:<12s}")
    if r["status"] == "SUCCESS":
        total_ok += 1

print(f"\n  {total_ok}/{len(pipeline_results)} stages succeeded")
print(f"\n  Final output: {os.path.join(PHASE2_DIR, 'predictions_output.parquet')}")
print(f"                {os.path.join(PHASE2_DIR, 'predictions_output.csv')}")



STAGE SUMMARY
  Stage  Name                      Status     Duration    
  ────────────────────────────────────────────────────────────
  [02]  Rules Engine              ✓ SUCCESS  0:01:34     
  [03]  Feature Engineering       ✓ SUCCESS  0:06:01     
  [04]  Phase 1 (predict)         ✓ SUCCESS  0:00:16     
  [05]  Phase 2 (predict)         ✓ SUCCESS  0:00:18     

  4/4 stages succeeded

  Final output: c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\phase2_outputs\predictions_output.parquet
                c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\phase2_outputs\predictions_output.csv


## 6 — Persistent Run History Log

In [6]:
# ════════════════════════════════════════════════════════════════
# PERSISTENT RUN LOG
# ════════════════════════════════════════════════════════════════
# Every pipeline run (training or inference) appends one JSON line to
# outputs_updated/run_history.jsonl. This file is the single source of
# truth for run history and grows monotonically — never overwritten.
#
# Each entry contains:
#   - run_id (timestamped, unique)
#   - timestamp (ISO 8601, local time)
#   - mode ("train" or "predict")
#   - pipeline_duration_seconds
#   - stage statuses (pass/fail per notebook)
#   - phase1 metrics  (AUC, F1, recall, threshold, ...)
#   - phase2 metrics  (primary/multi-label accuracy, threshold)
#   - row counts (input + predicted-AML if available)
#   - git_hash if .git is present
#
# Tail with:
#   python -c "import json; [print(json.dumps(json.loads(l), indent=2)) for l in open('outputs_updated/run_history.jsonl')]"
#
# Or load as a DataFrame:
#   pd.read_json("outputs_updated/run_history.jsonl", lines=True)
# ════════════════════════════════════════════════════════════════

import json as _json
import os as _os
import subprocess as _subprocess
from datetime import datetime as _datetime

def _safe_json_read(path):
    """Read a JSON file if it exists; return {} otherwise."""
    if not _os.path.exists(path):
        return {}
    try:
        with open(path) as _f:
            return _json.load(_f)
    except Exception:
        return {}

def _safe_parquet_row_count(path):
    """Count rows in a parquet file without loading it fully."""
    if not _os.path.exists(path):
        return None
    try:
        import pyarrow.parquet as _pq
        return int(_pq.ParquetFile(path).metadata.num_rows)
    except Exception:
        try:
            import pandas as _pd
            return int(len(_pd.read_parquet(path)))
        except Exception:
            return None

def _git_hash():
    try:
        out = _subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True, timeout=2)
        return out.stdout.strip() if out.returncode == 0 else None
    except Exception:
        return None

def write_run_log(mode, pipeline_results, pipeline_elapsed_seconds,
                  output_dir, phase1_dir, phase2_dir):
    """Append one entry to outputs_updated/run_history.jsonl."""
    now = _datetime.now()
    run_id = now.strftime("%Y%m%d_%H%M%S")

    # Phase 1 metrics — from model_parameters_full.json saved by 04
    p1_path = _os.path.join(phase1_dir, "model_parameters_full.json")
    p1_full = _safe_json_read(p1_path)
    p1_metrics = {}
    if "phase1" in p1_full:
        p1 = p1_full["phase1"]
        p1_metrics = {
            "best_config":     p1.get("best_config"),
            "threshold":       p1.get("threshold"),
            "auc_roc":         p1.get("auc_roc"),
            "f1_score":        p1.get("f1_score"),
            "precision":       p1.get("precision"),
            "recall":          p1.get("recall"),
            "tp":              p1.get("tp"),
            "fp":              p1.get("fp"),
            "fn":              p1.get("fn"),
            "tn":              p1.get("tn"),
            "best_iteration":  p1.get("best_iteration"),
            "n_features":      p1.get("n_features"),
            "n_train":         p1.get("n_train"),
            "n_test":          p1.get("n_test"),
            "imbalance_ratio": p1.get("imbalance_ratio"),
        }

    # Phase 2 metrics — from model_parameters_full.json saved by 05
    p2_path = _os.path.join(phase2_dir, "model_parameters_full.json")
    p2_full = _safe_json_read(p2_path)
    p2_metrics = {}
    if "phase2" in p2_full:
        p2 = p2_full["phase2"]
        p2_metrics = {
            "best_config":           p2.get("best_config"),
            "accuracy_primary":      p2.get("accuracy_primary"),
            "accuracy_multi_label":  p2.get("accuracy_multi_label"),
            "multi_label_threshold": p2.get("multi_label_threshold"),
            "macro_f1":              p2.get("macro_f1"),
            "weighted_f1":           p2.get("weighted_f1"),
            "best_iteration":        p2.get("best_iteration"),
            "n_classes":             p2.get("n_classes"),
            "n_features":            p2.get("n_features"),
            "n_train":                p2.get("n_train"),
            "n_test":                 p2.get("n_test"),
        }

    # Row counts — best-effort, parquets may not exist
    row_counts = {
        "rules_engine":        _safe_parquet_row_count(_os.path.join(output_dir, "stg_transactions_rules.parquet")),
        "feature_engineering": _safe_parquet_row_count(_os.path.join(output_dir, "stg_transactions_features.parquet")),
        "phase1_scored":       _safe_parquet_row_count(_os.path.join(phase1_dir, "df_ml_phase_1.parquet")),
        "phase2_predictions":  _safe_parquet_row_count(_os.path.join(phase2_dir, "predictions_output.parquet")),
    }
    # If in predict mode, the last parquet has the final predicted-AML count
    n_aml_predicted = None
    try:
        pred_path = _os.path.join(phase2_dir, "predictions_output.parquet")
        if _os.path.exists(pred_path):
            import pandas as _pd
            _pred = _pd.read_parquet(pred_path, columns=["predicted_typology"])
            n_aml_predicted = int((_pred["predicted_typology"].fillna("None") != "None").sum())
    except Exception:
        pass

    entry = {
        "run_id":      run_id,
        "timestamp":   now.isoformat(timespec="seconds"),
        "mode":        mode,
        "duration_seconds": round(pipeline_elapsed_seconds, 2),
        "duration_str":     str(_datetime.fromtimestamp(pipeline_elapsed_seconds, tz=None).strftime("%H:%M:%S")) if pipeline_elapsed_seconds > 0 else "—",
        "git_hash":    _git_hash(),
        "stages": [
            {"id": r["stage"], "name": r["name"], "status": r["status"], "duration": r.get("elapsed_str", "—")}
            for r in pipeline_results
        ],
        "stages_succeeded": sum(1 for r in pipeline_results if r["status"] == "SUCCESS"),
        "stages_total":     len(pipeline_results),
        "phase1_metrics":   p1_metrics,
        "phase2_metrics":   p2_metrics,
        "row_counts":       row_counts,
        "n_aml_predicted":  n_aml_predicted,
    }

    log_path = _os.path.join(output_dir, "run_history.jsonl")
    _os.makedirs(output_dir, exist_ok=True)
    with open(log_path, "a") as _f:
        _f.write(_json.dumps(entry, default=str) + "\n")

    # Pretty print summary
    print()
    print("=" * 70)
    print(f"RUN LOG — entry written to {log_path}")
    print("=" * 70)
    print(f"  Run ID:    {run_id}")
    print(f"  Mode:      {mode.upper()}")
    print(f"  Duration:  {entry['duration_seconds']}s")
    print(f"  Stages:    {entry['stages_succeeded']}/{entry['stages_total']} succeeded")
    if p1_metrics:
        print(f"  Phase 1:   AUC={p1_metrics.get('auc_roc'):.4f}  Recall={p1_metrics.get('recall'):.4f}  F1={p1_metrics.get('f1_score'):.4f}" if p1_metrics.get('auc_roc') else "  Phase 1:   (metrics unavailable)")
    if p2_metrics:
        ap = p2_metrics.get('accuracy_primary')
        am = p2_metrics.get('accuracy_multi_label')
        if ap and am:
            print(f"  Phase 2:   primary={ap*100:.2f}%  multi-label={am*100:.2f}%")
    if n_aml_predicted is not None:
        print(f"  Predicted AML rows: {n_aml_predicted:,}")
    print(f"  Total history entries: {sum(1 for _ in open(log_path))}")
    print("=" * 70)
    return log_path

# Call the writer — pipeline_results / pipeline_elapsed must be in scope.
# Determine mode automatically: training orchestrator has PIPELINE_MODE,
# inference orchestrator runs only in predict mode.
_log_mode = "train" if "PIPELINE_MODE" in dir() and PIPELINE_MODE == "train" else "predict"
_log_phase1_dir = _os.environ.get("AML_PHASE1_DIR",
    _os.path.join(_os.path.dirname(_os.getcwd()), "python_scripts", "ml_outputs"))
_log_phase2_dir = _os.environ.get("AML_PHASE2_DIR",
    _os.path.join(_os.path.dirname(_os.getcwd()), "python_scripts", "phase2_outputs"))

write_run_log(_log_mode, pipeline_results, pipeline_elapsed,
              OUTPUT_DIR, _log_phase1_dir, _log_phase2_dir)



RUN LOG — entry written to c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated\run_history.jsonl
  Run ID:    20260515_205537
  Mode:      PREDICT
  Duration:  491.45s
  Stages:    4/4 succeeded
  Phase 1:   AUC=0.9420  Recall=0.8562  F1=0.7476
  Phase 2:   primary=75.73%  multi-label=81.78%
  Predicted AML rows: 17,215
  Total history entries: 9


'c:\\Users\\VISHNUPRIYA\\OneDrive\\Desktop\\Freelancing\\AIGEN\\smartsentry_aml_model\\outputs_updated\\run_history.jsonl'

## 7 — Dashboard View *(monitoring summary)*

In [7]:
# ════════════════════════════════════════════════════════════════
# DASHBOARD VIEW — flatten run history into a monitoring table
# ════════════════════════════════════════════════════════════════
# Reads outputs_updated/run_history.jsonl (always present after the
# previous cell wrote a new entry). Builds / rebuilds a flat
# DataFrame with one row per run and saves it as
# outputs_updated/run_dashboard.csv — overwritten each time but
# always reflecting the COMPLETE history.
#
# Past runs are preserved by run_history.jsonl (append-only).
# The CSV is a derived view that's easy to open in Excel.
# ════════════════════════════════════════════════════════════════

import json as _json
import os as _os
import pandas as _pd


def update_run_dashboard(output_dir, recent_n=10):
    """Build / refresh the monitoring dashboard from run_history.jsonl.

    - If run_history.jsonl does not exist: returns an empty DataFrame.
    - If run_dashboard.csv does not exist: creates it.
    - If run_dashboard.csv already exists: overwrites with the full
      history (jsonl is the source of truth — CSV is a derived view).

    Parameters
    ----------
    output_dir : str
        Directory containing run_history.jsonl. Dashboard CSV is saved
        next to it.
    recent_n : int, default 10
        How many most-recent runs to print in the inline summary.

    Returns
    -------
    pandas.DataFrame
        One row per run, flattened. Empty if no history exists.
    """
    log_path  = _os.path.join(output_dir, "run_history.jsonl")
    dash_path = _os.path.join(output_dir, "run_dashboard.csv")

    # ─── Read every line from the JSONL log ────────────────────────
    if not _os.path.exists(log_path):
        print(f"⚠  No run history file at {log_path} — nothing to dashboard yet.")
        return _pd.DataFrame()

    rows = []
    with open(log_path) as _f:
        for line_no, raw in enumerate(_f, start=1):
            raw = raw.strip()
            if not raw:
                continue
            try:
                rows.append(_json.loads(raw))
            except _json.JSONDecodeError as exc:
                print(f"⚠  Skipping malformed log line {line_no}: {exc}")
                continue

    if not rows:
        print(f"⚠  Log file at {log_path} is empty.")
        return _pd.DataFrame()

    # ─── Flatten each entry into a single row ─────────────────────
    flat = []
    for r in rows:
        p1 = r.get("phase1_metrics") or {}
        p2 = r.get("phase2_metrics") or {}
        rc = r.get("row_counts") or {}
        flat.append({
            "run_id":            r.get("run_id"),
            "timestamp":         r.get("timestamp"),
            "mode":              r.get("mode"),
            "duration_seconds":  r.get("duration_seconds"),
            "stages_succeeded":  r.get("stages_succeeded"),
            "stages_total":      r.get("stages_total"),
            "git_hash":          r.get("git_hash"),
            # Phase 1
            "p1_best_config":    p1.get("best_config"),
            "p1_threshold":      p1.get("threshold"),
            "p1_auc_roc":        p1.get("auc_roc"),
            "p1_f1_score":       p1.get("f1_score"),
            "p1_precision":      p1.get("precision"),
            "p1_recall":         p1.get("recall"),
            "p1_tp":             p1.get("tp"),
            "p1_fp":             p1.get("fp"),
            "p1_fn":             p1.get("fn"),
            "p1_tn":             p1.get("tn"),
            "p1_n_features":     p1.get("n_features"),
            "p1_n_train":        p1.get("n_train"),
            "p1_n_test":         p1.get("n_test"),
            "p1_imbalance":      p1.get("imbalance_ratio"),
            # Phase 2
            "p2_best_config":         p2.get("best_config"),
            "p2_accuracy_primary":    p2.get("accuracy_primary"),
            "p2_accuracy_multilabel": p2.get("accuracy_multi_label"),
            "p2_multilabel_thresh":   p2.get("multi_label_threshold"),
            "p2_macro_f1":            p2.get("macro_f1"),
            "p2_weighted_f1":         p2.get("weighted_f1"),
            "p2_n_classes":           p2.get("n_classes"),
            "p2_n_train":             p2.get("n_train"),
            "p2_n_test":              p2.get("n_test"),
            # Row counts + final prediction count
            "n_rules":                 rc.get("rules_engine"),
            "n_features_eng":          rc.get("feature_engineering"),
            "n_phase1_scored":         rc.get("phase1_scored"),
            "n_phase2_predictions":    rc.get("phase2_predictions"),
            "n_aml_predicted":         r.get("n_aml_predicted"),
        })

    df_dash = _pd.DataFrame(flat)

    # Sort newest-first for display; CSV stays chronological
    df_dash_csv = df_dash.sort_values("timestamp", ascending=True).reset_index(drop=True)

    # ─── Write CSV (create or overwrite — full history every time) ─
    new_file = not _os.path.exists(dash_path)
    df_dash_csv.to_csv(dash_path, index=False)
    action = "Created" if new_file else "Refreshed"
    print(f"{action} dashboard CSV:")
    print(f"  Path:     {dash_path}")
    print(f"  Size:     {_os.path.getsize(dash_path)/1024:.1f} KB")
    print(f"  Runs:     {len(df_dash_csv)}")
    print(f"  Columns:  {len(df_dash_csv.columns)}")

    # ─── Recent runs summary (printed inline) ──────────────────────
    df_recent = df_dash.sort_values("timestamp", ascending=False).head(recent_n).copy()
    cols_to_show = [
        "run_id", "mode", "duration_seconds",
        "p1_auc_roc", "p1_recall", "p1_f1_score",
        "p2_accuracy_primary", "p2_accuracy_multilabel",
        "n_aml_predicted",
    ]
    df_show = df_recent[cols_to_show].copy()

    # Round numeric columns for readability
    for c in ["p1_auc_roc", "p1_recall", "p1_f1_score",
              "p2_accuracy_primary", "p2_accuracy_multilabel"]:
        if c in df_show.columns:
            df_show[c] = df_show[c].apply(lambda v: f"{v:.4f}" if _pd.notna(v) else "—")
    df_show["duration_seconds"] = df_show["duration_seconds"].apply(
        lambda v: f"{v:.1f}s" if _pd.notna(v) else "—")
    df_show["n_aml_predicted"] = df_show["n_aml_predicted"].apply(
        lambda v: f"{int(v):,}" if _pd.notna(v) else "—")

    print()
    print("=" * 95)
    print(f"DASHBOARD — last {min(recent_n, len(df_dash))} run(s) (newest first)")
    print("=" * 95)
    print(df_show.to_string(index=False))
    print("=" * 95)

    # ─── Quick aggregate stats (mean / std over all training runs) ─
    train_mask = df_dash["mode"] == "train"
    if train_mask.sum() >= 2:
        train_df = df_dash[train_mask]
        print()
        print("Training-run aggregates (mean ± std across all training runs):")
        for col, label in [
            ("p1_auc_roc",            "Phase 1 AUC-ROC"),
            ("p1_recall",             "Phase 1 Recall"),
            ("p1_f1_score",           "Phase 1 F1"),
            ("p2_accuracy_primary",   "Phase 2 primary acc"),
            ("p2_accuracy_multilabel","Phase 2 multi-label acc"),
        ]:
            vals = _pd.to_numeric(train_df[col], errors="coerce").dropna()
            if len(vals) >= 2:
                print(f"  {label:<28s} {vals.mean():.4f} ± {vals.std():.4f}  (n={len(vals)})")

    return df_dash


# ─── Run it ────────────────────────────────────────────────────────
_dashboard_df = update_run_dashboard(OUTPUT_DIR, recent_n=10)


Refreshed dashboard CSV:
  Path:     c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated\run_dashboard.csv
  Size:     3.5 KB
  Runs:     9
  Columns:  35

DASHBOARD — last 9 run(s) (newest first)
         run_id    mode duration_seconds p1_auc_roc p1_recall p1_f1_score p2_accuracy_primary p2_accuracy_multilabel n_aml_predicted
20260515_205537 predict           491.4s     0.9420    0.8562      0.7476              0.7573                 0.8178          17,215
20260515_203620 predict           212.4s     0.9420    0.8562      0.7476              0.7573                 0.8178          17,215
20260515_203113 predict            56.9s     0.9420    0.8562      0.7476              0.7573                 0.8178          17,215
20260515_202625 predict            86.1s     0.9420    0.8562      0.7476              0.7573                 0.8178          17,215
20260515_192813 predict           555.3s     0.9420    0.8562      0.7476              0.7573   